# 00 · 환경 점검 & GPU 컴퓨팅 / CuPy 개론

> **CuPy 2일 집중 코스 — Day 1 / 단원 1 (GPU 컴퓨팅과 CuPy 개론)**

이 노트북은 코스의 출발점입니다. GPU가 *왜* 빠른지, CuPy가 *무엇*인지 개념을 잡고,
실습 환경을 점검한 뒤 첫 GPU 연산을 실행합니다.

### 왜 GPU 컴퓨팅을 배우는가
딥러닝·과학계산·데이터분석 워크로드는 대부분 **같은 연산을 대량의 데이터에 반복 적용**하는 형태입니다
(행렬곱, 원소별 연산, FFT, 리덕션 등). 이런 워크로드는 코어 수가 훨씬 많은 GPU에서 수 배~수십 배
빨라질 수 있습니다. 문제는 "GPU가 빠르다"는 사실 자체가 아니라 **언제, 어떻게 빠른가**를 정확히
아는 것입니다 — 이 코스 전체가 그 질문에 답합니다. 00은 그 첫 단추로, 하드웨어 관점의 큰 그림과
가장 흔한 실수(비동기 착시, 불필요한 전송)를 미리 짚고 갑니다.

### 이 노트북의 흐름
개념(CPU vs GPU → 프로그래밍 모델 → CuPy 소개) → 환경 점검 → 첫 실행 → **비동기·전송이라는
두 가지 특징 체험** → 포팅 연습. 이 순서는 이후 모든 노트북에서 반복되는 패턴
("이론 → 실습 → 연습")의 원형입니다.

## 학습 목표
- CPU와 GPU의 구조적 차이(지연시간 vs 처리량)를 설명할 수 있다.
- GPU 프로그래밍 모델(host/device, 커널, grid·block·thread, SIMT)의 큰 그림을 안다.
- CuPy가 NumPy/SciPy의 드롭인 대체임을 이해하고 첫 연산을 실행한다.
- **GPU 연산은 비동기**라는 사실과, 올바른 시간 측정·전송 비용을 체감한다.
- `course_utils`의 측정 유틸(`bench`, `print_env` 등)이 왜 필요한지 이해하고 이후 노트북에서
  반복 사용할 준비를 한다.


## 목차 (Table of Contents)
1. [CPU vs GPU — 왜 GPU인가](#1)
2. [GPU 프로그래밍 모델](#2)
3. [CuPy란 무엇인가](#3)
   - [3.1 CuPy ≠ NumPy 차이점](#3-1)
   - [3.2 CuPy 루틴 지도](#3-2)
4. [실습 환경 설정 & 점검](#4)
5. [첫 GPU 연산](#5)
6. [비동기 타이밍의 함정](#6)
   - [6.1 일회성 오버헤드와 워밍업](#6)
7. [전송 비용 (host ↔ device)](#7)
8. [연습문제](#8)
9. [체크포인트](#9)

### 실습 규칙 (중요)

이 세 가지는 코스 전체에서 반복적으로 등장하는, GPU 코드를 처음 짤 때 가장 흔히 저지르는 실수입니다.
지금 이유를 이해해두면 이후 노트북에서 같은 규칙을 볼 때마다 낯설지 않습니다.

- **비동기(async) 시간 측정**: GPU 연산은 기본적으로 **비동기**입니다. 파이썬 코드가 커널을
  '제출'하는 순간과 GPU가 실제로 '완료'하는 순간이 다르기 때문에, 일반 타이머로 감싸면
  실제보다 훨씬 빠르게(심하면 거의 0에 가깝게) 측정되는 착시가 생깁니다. 반드시
  **동기화(synchronize) 후** 시간을 재거나, 동기화를 자동으로 처리하는 `course_utils.bench`
  (CUDA 이벤트 기반)를 사용하세요. → 6절에서 직접 눈으로 확인합니다.
- **전송(transfer) 최소화**: `cp.asnumpy()`(device→host)와 `cp.asarray()`(host→device)는
  PCIe 버스를 통한 물리적 데이터 복사이며, GPU 연산 자체보다 **훨씬 느릴 수 있습니다**.
  반복문 안에서 매번 호출하면 GPU 가속 효과가 전송 비용에 묻혀 사라집니다. 연산은 GPU에
  머무르게 하고, 결과 출력·플롯 등 **정말 필요한 마지막 순간에만** 전송하세요. → 7절에서 수치로 확인합니다.
- **정확성 검증**: GPU는 연산 순서가 CPU와 달라 부동소수점 결과가 **마지막 비트까지 완전히
  같지는 않습니다**(부동소수점 덧셈은 결합법칙이 엄밀히 성립하지 않기 때문). 따라서 `==` 대신
  `numpy.testing.assert_allclose`로 허용오차(`rtol`/`atol`) 내에서 같은지 검증합니다. float32는
  float64보다 오차가 크므로 허용오차를 약간 더 키우는 것이 일반적입니다.


<a id="1"></a>
## 1. CPU vs GPU — 왜 GPU인가

CPU와 GPU는 **설계 철학**이 다릅니다.

| 구분 | CPU | GPU |
|------|-----|-----|
| 코어 | 소수(수~수십 개)의 **강력한** 코어 | 수천 개(1,000~10,000+)의 **단순한** 코어 |
| 최적화 목표 | **지연시간(latency)** 최소화 — 한 작업을 빨리 | **처리량(throughput)** 최대화 — 많은 작업을 동시에 |
| 메모리 대역폭 | 대략 ~100 GB/s | 대략 500 GB/s ~ 2 TB/s |
| 강점 | 분기 많은 순차 로직, OS/DB, 제어 흐름 | 대규모 **데이터 병렬** 연산(행렬, 신호, 이미지) |

GPU는 개별 코어의 클럭은 낮지만, **같은 연산을 거대한 데이터에 동시에** 적용하는 작업에서 압도적입니다.
NumPy의 `a * b`처럼 원소별·벡터화된 연산이 바로 그런 형태라 GPU 가속의 대상이 됩니다.

> GPU는 '한 명의 천재'가 아니라 '수십 명의 일꾼'입니다. 일을 잘게 나눠 동시에 줄 수 있을 때 빛납니다.
> 반대로 작은 배열이나 순차 의존이 강한 작업은 GPU가 오히려 느릴 수 있습니다(런치 오버헤드 + 전송 비용).

### 조금 더 구체적으로: 지연시간 vs 처리량

- **지연시간(latency)**: 작업 하나를 시작해서 끝내는 데 걸리는 시간. CPU는 코어당 큰 캐시·분기예측·
  비순차 실행(out-of-order execution) 등 정교한 회로를 동원해 **하나의 작업을 최대한 빨리** 끝내도록 설계됩니다.
- **처리량(throughput)**: 단위 시간에 처리하는 작업의 총량. GPU는 개별 코어의 회로를 단순화하는 대신
  그 자리에 코어를 훨씬 많이 채워, **여러 작업을 동시에** 처리해 총량을 늘립니다.

비유하면 CPU는 '고급 스포츠카 한 대'(빠르지만 한 번에 한 명), GPU는 '수십 인승 고속버스 함대'(개별 속도는
평범하지만 한 번에 실어나르는 승객 수가 압도적)에 가깝습니다. 승객(데이터)이 적으면 스포츠카가 더 빠르지만,
승객이 아주 많으면 버스 함대가 총 이동 시간에서 압도적으로 유리합니다 — 이것이 "문제 크기가 작으면 GPU가
오히려 느릴 수 있다"는 현상의 근원이며, `01_benchmark_basics`에서 **손익분기점**으로 정량화합니다.

메모리 대역폭 차이(~100 GB/s vs 500 GB/s~2 TB/s)도 같은 맥락입니다. GPU 연산이 아무리 빨라도 데이터를
공급하는 속도(대역폭)가 부족하면 코어는 놀게 됩니다 — 연산이 **메모리 병목(memory-bound)** 인지
**연산 병목(compute-bound)** 인지 구분하는 사고방식은 `05_memory_profiling`에서 본격적으로 다룹니다.


<a id="2"></a>
## 2. GPU 프로그래밍 모델

**Host(CPU)와 Device(GPU)는 서로 다른 메모리 공간**을 가집니다. 큰 그림은 다음과 같습니다.
1. host 메모리(NumPy)에서 device 메모리(CuPy 배열)로 데이터를 **전송**한다.
2. device에서 **커널(kernel)** — GPU에서 실행되는 함수 — 을 실행해 연산한다.
3. 필요한 결과만 다시 host로 **전송**해 가져온다.

커널은 수많은 **스레드(thread)** 로 실행됩니다. 스레드는 다음 계층으로 조직됩니다.
- **thread** → 가장 작은 실행 단위(보통 원소 1개 담당)
- **block** → 스레드들의 묶음(공유 메모리를 공유)
- **grid** → 블록들의 묶음(전체 문제 공간)

하드웨어는 스레드를 **warp(32개)** 단위로 묶어 **SIMT**(Single Instruction, Multiple Threads) — *한 명령을 여러 스레드가 동시에* — 방식으로 실행합니다.

> 좋은 소식: CuPy로 NumPy 코드를 포팅할 때는 이 grid/block/thread를 **직접 다루지 않아도** 됩니다.
> CuPy가 알아서 커널을 생성·실행합니다. 직접 제어는 Day 2의 커널 작성에서 배웁니다.

### 조금 더 구체적으로

- **thread**: 커널 코드 한 벌을 실행하는 최소 단위. 보통 배열의 원소 하나(또는 몇 개)를 담당합니다.
- **block**: 수십~수천 개의 thread 묶음. 같은 block 안의 thread끼리는 **공유 메모리(shared memory)**
  와 **동기화(`__syncthreads`)** 를 사용할 수 있습니다. 블록 크기는 보통 128~1024 사이에서 선택합니다.
- **grid**: 전체 문제를 덮기 위한 block들의 묶음. 배열이 커지면 grid도 커져, 수백만 개의 thread가
  동시에(논리적으로) 실행될 수 있습니다.
- **warp**: 하드웨어가 thread를 32개씩 묶어 관리하는 단위. 같은 warp의 32개 thread는 **정확히 같은
  명령어를 동시에** 실행합니다(SIMT). 이 때문에 warp 안에서 조건문(`if`)에 따라 다른 코드 경로를
  타면(**warp divergence**) 두 경로를 순차적으로 실행하게 되어 성능이 떨어집니다 — Day 2 커널
  최적화(`07`~`11`)에서 다시 등장하는 핵심 개념입니다.

host↔device 데이터 전송은 물리적으로 **PCIe 버스**(또는 NVLink)를 통해 이뤄지며, 이는 GPU 내부
메모리 대역폭보다 한 자릿수 이상 느립니다. 커널 실행(launch) 자체도 매번 마이크로초 단위의
고정 오버헤드가 있어, 아주 작은 연산을 GPU로 보내면 오버헤드가 실제 연산 시간을 압도할 수 있습니다.


호스트(CPU)에서는 **모든 파이썬**(객체, 클래스, 예외, 동적 타입 등)이 자유롭게 동작하지만,
디바이스(GPU)에서 실행되는 커널 코드는 **파이썬의 일부(subset)** 만 사용할 수 있습니다. 예를 들어
커널 안에서는 리스트/딕셔너리 같은 파이썬 객체 생성, 임의 함수 호출, 예외 처리(try/except),
동적 타입 변경 등이 보통 제한되고, 대신 정해진 수치 연산과 배열 인덱싱 위주로 작성해야 합니다
(정확한 규칙은 Day 2 `08_numba_copy`의 `@cuda.jit`에서 다룹니다).

**Day 1에서는 이 제약을 직접 신경 쓸 필요가 없습니다.** `cp.sin(x)` 같은 CuPy 함수를 호출하면
CuPy가 내부적으로 이미 준비된 CUDA 커널을 실행해주기 때문입니다. 사용자가 커널 코드를 직접
작성하는 것은 Day 2(`07`~`11`)의 주제입니다.

<img src="images/figures/new_host_device_code.png" width="600">

<sub>그림: 호스트 코드(자유로운 파이썬)와 디바이스 코드(제한된 서브셋)의 대비</sub>


<a id="3"></a>
## 3. CuPy란 무엇인가

**CuPy는 GPU에서 동작하는 NumPy/SciPy 라이브러리**로, 기존 코드를 NVIDIA CUDA(또는 AMD ROCm)에서 실행하는 **드롭인 대체(drop-in replacement)** 를 지향합니다.

- NumPy/SciPy와 **약 90% 호환** — `import numpy as np`를 `import cupy as cp`로 바꾸면 대부분 그대로 동작합니다.
- **메모리 할당과 GPU 커널 실행을 CuPy가 대신** 처리합니다(사용자가 CUDA를 직접 쓰지 않아도 됨).
- 오픈소스이며 **NVIDIA CUDA & AMD ROCm** 백엔드를 지원합니다.
- 내부적으로 NVIDIA 검증 라이브러리로 구동: **cuBLAS**(선형대수), **cuFFT**(FFT), **cuSPARSE**(희소행렬), **cuSOLVER**, **cuRAND**(난수), Thrust/CUB 등.
- **상호운용 표준** 지원: DLPack, CUDA Array Interface, `__array_function__` 등 → PyTorch·TensorFlow 등과 무복사 연동.
- 필요하면 **사용자 정의 CUDA 커널**(Elementwise/Reduction/RawKernel)도 작성 가능(Day 2).

> NVIDIA GTC 자료 기준, **거의 동일한 코드로 약 10배 빠른** 사례가 보고됩니다(예: Quadro RTX 8000). 다만 가속 폭은 문제 크기·하드웨어에 따라 달라집니다.

**조금 더 배경**: CuPy는 원래 딥러닝 프레임워크 Chainer를 위한 GPU 배열 라이브러리로 시작되었고,
이후 Chainer와 분리되어 독립적인 오픈소스 프로젝트로 발전했습니다. 지금은 NumPy/SciPy API를 최대한
그대로 재현하는 것을 목표로 활발히 관리되고 있습니다.

**"거의 같다"의 의미**: 함수 시그니처와 동작이 대부분 동일하다는 뜻이지, 성능·정밀도·엣지케이스까지
동일하다는 뜻은 아닙니다. 다음 절(3.1)에서 실습 중 마주칠 수 있는 구체적인 차이를 짚습니다. 또한
NumPy 자체의 `__array_function__`(NEP 18) 디스패치 메커니즘 덕분에, 경우에 따라 `cp.`로 바꾸지
않고도 NumPy 함수에 CuPy 배열을 그대로 넘기면 자동으로 GPU 버전이 호출되기도 합니다
(`02_ndarray_core`에서 `get_array_module`과 함께 다룹니다).

```python
# NumPy
import numpy as np
x = np.arange(1_000_000)
y = np.sin(x).sum()

# CuPy — 거의 동일!
import cupy as cp
x = cp.arange(1_000_000)
y = cp.sin(x).sum()   # GPU에서 실행
```


<a id="3-1"></a>
### 3.1 CuPy ≠ NumPy: 알아둘 차이점

"거의 같다"지만 **똑같지는 않습니다.** 다음 차이는 실습 중 버그로 이어지기 쉬우니 미리 알아둡니다.
(출처: NVIDIA GTC 강의자료 + CuPy 공식 문서 *Differences between CuPy and NumPy* — 두 출처에서 교차확인)

| 항목 | NumPy | CuPy |
|------|-------|------|
| 함수 커버리지 | 전체 | **대부분 지원**(일부 함수 없음) |
| dtype | 문자열·object·구조체 가능 | **숫자형 위주**(문자열/object 미지원, 구조체 매우 제한적) |
| 경계 밖 정수 인덱싱 | **에러(IndexError)** | **wrap-around** (조용히 처리 → 버그 주의!) |
| 메모리 전송 | 불필요 | 다른 라이브러리(OpenCV·matplotlib 등)와 주고받으려면 **명시적 전송 필요** |
| 타입 승격/캐스팅 | 안전하지만 느릴 수 있음 | **속도 우선** — 일부 캐스팅 결과가 다름(예: 음수 float→uint) |
| 난수 | — | 알고리즘이 달라 **bit 단위로 동일하지 않음**(cuRAND) |
| 리덕션 결과 | 스칼라(`np.float32`) | **0-차원 `cupy.ndarray`** (불필요한 동기화 회피 위함; 스칼라가 필요하면 `float()`/`.item()`) |
| ufunc 입력 | list·np.ndarray도 허용 | **CuPy 배열/스칼라만** 허용 |

표에서 특히 실습 중 자주 걸리는 두 가지를 조금 더 풀어봅니다.

- **경계 밖 인덱싱 wrap-around**: NumPy는 `a[len(a)]`처럼 범위를 벗어나면 즉시 `IndexError`를
  던져 버그를 바로 알 수 있지만, CuPy는 조용히 음수 인덱스처럼 순환(wrap-around)해 **잘못된 값을
  반환하고도 에러가 나지 않을 수 있습니다.** 인덱스 계산 실수를 놓치기 쉬우니 인덱스 범위는
  스스로 검증하는 습관이 필요합니다.
- **리덕션 결과가 0차원 배열**: `x.sum()`이 NumPy에서는 파이썬/NumPy 스칼라를 반환하지만, CuPy에서는
  **0차원 `cupy.ndarray`** 를 반환합니다. 이는 즉시 host 스칼라로 변환(=동기화)하지 않고 **GPU에
  계속 남겨두어** 불필요한 동기화를 피하기 위한 설계입니다. 실제 파이썬 숫자가 필요하면
  `float(x.sum())` 또는 `x.sum().item()`처럼 명시적으로 변환하세요.

> ⚠️ **직렬 for 루프를 피하세요.** 파이썬 `for`로 원소를 하나씩 처리하면 GPU에서는 CPU보다 **수십~수백 배 느려질 수 있습니다.**
> 항상 **벡터화된 배열 연산**(`a * b + c`, `cp.sum`, 슬라이싱)으로 표현하세요. 이것이 GPU 가속의 핵심 전제입니다.


<a id="3-2"></a>
### 3.2 CuPy 루틴 지도 (공식 overview)

CuPy 공식 [overview](https://docs.cupy.dev/en/stable/overview.html)는 라이브러리를 **4대 구성**으로 소개합니다. 각 루틴은 NVIDIA CUDA 라이브러리가 구동합니다.

| 구성 | 모듈 (공식 레퍼런스) | 백엔드 |
|------|----------------------|--------|
| **N차원 배열** | [`cupy.ndarray`](https://docs.cupy.dev/en/stable/reference/ndarray.html) | — |
| **NumPy 루틴** | [`cupy.*`](https://docs.cupy.dev/en/stable/reference/routines.html) · [`linalg`](https://docs.cupy.dev/en/stable/reference/linalg.html) · [`fft`](https://docs.cupy.dev/en/stable/reference/fft.html) · [`random`](https://docs.cupy.dev/en/stable/reference/random.html) | cuBLAS·cuSOLVER·cuFFT·cuRAND |
| **SciPy 루틴** | [`fft`](https://docs.cupy.dev/en/stable/reference/scipy_fft.html) · [`linalg`](https://docs.cupy.dev/en/stable/reference/scipy_linalg.html) · [`ndimage`](https://docs.cupy.dev/en/stable/reference/scipy_ndimage.html) · [`special`](https://docs.cupy.dev/en/stable/reference/scipy_special.html) · [`signal`](https://docs.cupy.dev/en/stable/reference/scipy_signal.html) · [`stats`](https://docs.cupy.dev/en/stable/reference/scipy_stats.html) | cuFFT·cuSOLVER 등 |
| **희소행렬** | [`cupyx.scipy.sparse`](https://docs.cupy.dev/en/stable/reference/scipy_sparse.html) · [`sparse.linalg`](https://docs.cupy.dev/en/stable/reference/scipy_sparse_linalg.html) | cuSPARSE |

또한 직접 [커스텀 CUDA 커널](https://docs.cupy.dev/en/stable/user_guide/kernel.html)(Elementwise/Reduction/Raw/JIT/Fusion)을 작성할 수 있고(→ Day 2),
DLPack·CUDA Array Interface 등 표준으로 PyTorch·TensorFlow와 [상호운용](https://docs.cupy.dev/en/stable/user_guide/interoperability.html)됩니다(→ 단원 7).

> **참고**: SciPy에 대응하는 기능은 `cupyx.scipy.*` 네임스페이스에 있습니다(SciPy 자체가 아니라
> "CuPy 확장"이라는 의미의 `cupyx` 접두어). 반면 NumPy에 대응하는 기능은 `cupy.*`에 바로 있습니다 —
> 이 네이밍 차이를 기억해두면 `04_scipy_routines`에서 import 실수를 줄일 수 있습니다.

> **코스 매핑**: ndarray→`02`, NumPy 루틴→`03`, SciPy 루틴→`04`, 메모리/스트림→`05`/`06`, 커널→Day 2.


<a id="4"></a>
## 4. 실습 환경 설정 & 점검

CuPy는 **설치된 CUDA 툴킷 버전에 맞는 바이너리(wheel)** 를 선택해야 합니다. 버전이 맞지 않으면
`import cupy`부터 실패할 수 있으므로, 설치 전 `nvidia-smi`로 드라이버가 지원하는 CUDA 버전을
먼저 확인해두면 좋습니다.

`pip`으로 설치하는 경우 (CUDA 버전에 맞춰 **둘 중 하나만** 설치):
```bash
pip install cupy-cuda13x      # CUDA 13.x
# 또는
pip install cupy-cuda12x      # CUDA 12.x
pip install numpy scipy matplotlib
```

이 실습 환경에서는 아래 셀처럼 **conda(conda-forge)** 로 CuPy와 Day 2에서 필요한
`numba`·`cuda-cccl`을 한 번에 설치합니다. conda는 CUDA 관련 런타임 라이브러리까지 함께
관리해주어 버전 충돌 가능성을 줄여줍니다.


In [4]:
! conda install --override-channels -c conda-forge cupy numba cuda-cccl[cu13] "numpy<2.4" scipy matplotlib -y

Channels:
 - conda-forge
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /opt/conda

  added / updated specs:
    - cuda-cccl
    - cupy
    - matplotlib
    - numba
    - numpy[version='<2.4']
    - scipy


The following NEW packages will be INSTALLED:

  alsa-lib           conda-forge/linux-64::alsa-lib-1.2.16.1-hb03c661_0 
  brotli             conda-forge/linux-64::brotli-1.2.0-hed03a55_1 
  brotli-bin         conda-forge/linux-64::brotli-bin-1.2.0-hb03c661_1 
  cairo              conda-forge/linux-64::cairo-1.18.4-he90730b_1 
  cccl               conda-forge/linux-64::cccl-3.3.4-hd4ab2ea_0 
  contourpy          conda-forge/linux-64::contourpy-1.3.3-py313hc8edb43_4 
  cuda-cccl          conda-forge/linux-64::cuda-cccl-13.3.3.4.1-ha770c72_1 
  cuda-cccl_linux-64 conda-forge/noarch::cuda-cccl_linux-64-13.3.3.4.1-ha770c72_1 
  cuda-cudart-dev_l~ conda-forge/noarch::cuda-cudart-dev_linux-64-13.3.29-h376f20c_0 
  cuda-cudart-stati~ conda-forge/no

아래 셀은 이번 코스에서 계속 재사용할 **공통 셋업 패턴**입니다: 표준 라이브러리 import →
`cupy` import(실패 시 즉시 원인과 함께 에러) → 작업 디렉터리 이동(상대경로로 이미지·데이터 참조가
항상 같은 위치를 보도록) → `course_utils`에서 이번 노트북에 필요한 함수만 import →
`print_env()`로 환경 확인. **GPU 이름과 VRAM이 정상 출력되면 준비 완료**이고, `cupy` import
자체가 실패한다면 위 4절의 설치 과정을 다시 확인하세요.


In [2]:
# 공통 셋업: 패키지 + 코스 유틸리티 로드
import os, sys, time, math
import numpy as np

try:
    import cupy as cp
except Exception as e:
    raise RuntimeError(
        "CuPy import 실패. cupy-cuda13x 또는 cupy-cuda12x를 설치하세요.\n"
        f"원인: {e}"
    ) from e

os.chdir(os.path.expanduser("/root/cupy"))   # 홈 디렉터리로 이동

from course_utils import print_env, bytes_human, bench, gpu_ms, cpu_ms

print_env()

=== Environment ===
numpy: 2.3.5
cupy : 14.1.1
device: 0 - NVIDIA A100-SXM4-80GB (compute capability 8.0)
VRAM  : 79.25 GB


### 📦 `course_utils` 함수 — 이번 노트북에서 도입

실습 공통 유틸리티는 `course_utils.py`에 모아두고 노트북이 진행되며 **필요한 함수만 하나씩**
추가합니다. 00에서 도입하는 함수:

- **`bytes_human(n)`** — 바이트 수를 `KB/MB/GB` 문자열로 변환(메모리 크기 출력용).
- **`print_env()`** — numpy·cupy 버전, GPU 이름, VRAM을 한 번에 출력(바로 위 셀에서 실행).
- **`bench(fn, n_repeat=20, n_warmup=3)`** — `cupyx.profiler.benchmark` 래퍼. 워밍업·동기화·반복평균을 자동 처리해 **비동기 GPU를 올바르게 측정**합니다.
- **`gpu_ms(r)` / `cpu_ms(r)`** — `bench` 결과에서 GPU 커널 / CPU wall-clock 평균 시간(ms)을 추출.

> 아래 코드 셀들은 `course_utils.py`에 있는 실제 구현을 **그대로 가져와 보여주는 것**입니다
> (교육 목적의 노출). 실전 코드에서는 `from course_utils import ...`로 가져다 쓰면 되고, 매
> 노트북마다 다시 정의할 필요는 없습니다. 각 함수 안의 주석을 함께 읽으면 "왜 이렇게 구현했는지"까지
> 파악할 수 있습니다.


In [3]:
def bytes_human(n: int) -> str:
    """바이트 수를 KB/MB/GB 등 사람이 읽기 쉬운 문자열로 변환.

    예: bytes_human(1536) -> "1.50 KB"
    예: bytes_human(2**30) -> "1.00 GB"
    """
    # 1024배씩 커지는 단위 목록. GPU/OS 메모리 표기 관례에 맞춰 1000이 아닌 1024(2^10) 기준.
    suffixes = ["B", "KB", "MB", "GB", "TB"]
    x = float(n)  # 정수 나눗셈으로 인한 반올림 손실을 피하기 위해 float로 변환
    for s in suffixes:
        # x가 1024 미만이면 더 이상 올릴 단위가 없으므로 현재 단위(s)로 확정해 반환
        # (또는 마지막 단위인 TB까지 왔다면 그 이상은 없으므로 무조건 반환)
        if x < 1024.0 or s == suffixes[-1]:
            return f"{x:.2f} {s}"
        # 아직 1024 이상이면 한 단계 큰 단위로 넘어가기 위해 1024로 나눔 (B->KB->MB->GB->TB)
        x /= 1024.0
    # 방어적 코드: 위 루프는 suffixes[-1]("TB")에서 항상 return하므로 실제로는 도달하지 않음
    return f"{x:.2f} TB"


In [4]:
def print_env() -> None:
    """실습 환경 요약 출력: numpy/cupy 버전, GPU 이름, VRAM."""
    print("=== Environment ===")
    print("numpy:", np.__version__)  # numpy는 필수 의존성이므로 항상 설치되어 있다고 가정
    if cp is None:
        # 이 모듈 상단의 `try: import cupy as cp / except: cp = None`에 의해
        # CuPy 미설치·미가용 환경에서는 cp가 None이 됨 -> 여기서 안내만 하고 조용히 종료
        print("cupy : (사용 불가) — cupy-cuda12x 또는 cupy-cuda11x 설치 필요")
        print("===================")
        return  # GPU가 없으므로 아래 디바이스 조회 로직은 의미가 없어 함수 종료
    print("cupy :", cp.__version__)
    try:
        dev = cp.cuda.Device()  # 현재 활성(디폴트) CUDA 디바이스 핸들(보통 0번)
        # CUDA 런타임 API로 하드웨어 속성(이름, compute capability, 총 VRAM 등)을 조회
        props = cp.cuda.runtime.getDeviceProperties(dev.id)
        name = props["name"]
        # CUDA 버전/플랫폼에 따라 이름이 bytes로 오는 경우가 있어 문자열로 디코딩 처리
        name = name.decode() if isinstance(name, (bytes, bytearray)) else name
        # compute capability: GPU 아키텍처 세대를 나타내는 버전(예: 8.0 = Ampere, 9.0 = Hopper)
        cc = f"{props['major']}.{props['minor']}"
        total_mem = int(props.get("totalGlobalMem", 0))  # GPU 전체 VRAM 용량(바이트)
        print("device:", f"{dev.id} - {name} (compute capability {cc})")
        print("VRAM  :", bytes_human(total_mem))  # 위에서 정의한 bytes_human으로 가독성 좋게 변환
    except Exception as e:
        # 드라이버 미설치/권한 문제 등으로 조회가 실패해도 노트북 실행이 멈추지 않도록 예외 처리
        print("device: (조회 실패)", e)
    print("===================")


In [5]:
def bench(fn, *, n_repeat: int = 20, n_warmup: int = 3, name: str | None = None):
    """올바른 GPU 타이밍 헬퍼.

    GPU 연산은 비동기라서 time.perf_counter()로는 정확히 잴 수 없습니다.
    cupyx.profiler.benchmark는 CUDA 이벤트로 동기화하여 CPU/GPU 시간을 함께 측정합니다.

    Parameters
    ----------
    fn : Callable
        인자 없이 호출 가능한 벤치마크 대상 함수 (예: `lambda: cp.sin(x)`).
    n_repeat : int
        실제로 측정에 반영되는 반복 횟수. 이 반복들의 시간으로 평균/표준편차를 낸다.
    n_warmup : int
        측정에서 제외되는 예열(warm-up) 반복 횟수. CUDA 컨텍스트 초기화, 커널
        최초 컴파일/캐시 적재 등 '일회성 오버헤드'가 측정값을 왜곡하지 않도록 미리 소모한다.
    name : str, optional
        결과에 표시할 이름. 지정하지 않으면 fn.__name__을 사용(람다는 이름이 없어 "fn"이 됨).

    Returns
    -------
    cupyx.profiler._time._PerfCaseResult
        gpu_times / cpu_times (각 n_repeat 길이의 배열, 단위: 초)를 담고 있으며,
        이 모듈의 gpu_ms()/cpu_ms()로 평균값(ms)을 뽑아 쓴다.
    """
    if cp is None:
        # CuPy가 없으면 GPU 벤치마크 자체가 불가능하므로 조용히 넘어가지 않고 명확히 에러 발생
        raise RuntimeError("CuPy를 사용할 수 없습니다. cupy-cuda12x/11x를 설치하세요.")
    # benchmark는 각 반복마다 CUDA 이벤트를 커널 앞뒤로 기록하고, 이벤트를 동기화(sync)해
    # '커널이 실제로 끝난 시점'까지 기다린 뒤 시간을 재기 때문에 비동기 문제 없이 정확하다.
    from cupyx.profiler import benchmark
    return benchmark(
        fn, (),                                        # fn을 인자 없이(()) 반복 호출
        n_repeat=n_repeat,
        n_warmup=n_warmup,
        name=name or getattr(fn, "__name__", "fn"),     # 이름 미지정 시 함수명 사용
    )


In [6]:
def gpu_ms(result) -> float:
    """bench() 결과에서 GPU 평균 시간(ms).

    result.gpu_times: 각 반복에서 GPU가 실제로 커널을 실행한 시간(단위: 초)의 배열.
    CPU가 커널을 GPU에 '제출'만 하고 바로 다음 줄로 넘어가는 launch overhead는
    포함하지 않으므로, 순수 커널 연산 시간(GPU 관점)을 보고 싶을 때 사용한다.
    """
    # 여러 반복(n_repeat)의 평균을 낸 뒤, 초(s) 단위를 밀리초(ms)로 변환(x1e3)
    return float(np.asarray(result.gpu_times).mean()) * 1e3


def cpu_ms(result) -> float:
    """bench() 결과에서 CPU(런치 포함) 평균 시간(ms).

    result.cpu_times: 각 반복에서 '호출 시작~반환'까지 CPU 입장에서 흐른
    wall-clock 시간(단위: 초)의 배열. 커널 launch overhead와 파이썬 오버헤드,
    (필요 시) 동기화 대기까지 포함하므로 사용자가 체감하는 end-to-end 시간에 더 가깝다.
    """
    return float(np.asarray(result.cpu_times).mean()) * 1e3


<a id="5"></a>
## 5. 첫 GPU 연산

`cp.arange`, `cp.sin` 등은 NumPy와 동일하게 쓰되 **결과가 GPU 메모리에 만들어집니다**.
`x`는 파이썬 객체로는 `cupy.ndarray` 타입이며, `.device` 속성으로 어느 GPU(멀티 GPU 환경이라면
몇 번 디바이스)에 있는지 확인할 수 있습니다. 마지막에 `cp.asnumpy`로 host로 가져와 확인합니다
(이 한 번의 전송은 "결과 확인"을 위한 것이라 문제 없습니다 — 문제는 **반복문 안에서** 매번
전송하는 것입니다. 2절 실습 규칙을 떠올려 보세요).

내부적으로 `cp.sin(x)`를 처음 호출하면 CuPy가 이 shape/dtype에 맞는 CUDA 커널을 즉석에서
컴파일(JIT)하고 캐시합니다. 이 노트북 뒷부분(6.1절)에서 다루는 "일회성 오버헤드"가 사실
여기서부터 시작됩니다.


In [8]:
x = cp.arange(10, dtype=cp.float32)   # GPU에 배열 생성
y = cp.sin(x) + 2.0                    # GPU에서 계산 (커널 자동 생성/실행)

print('타입      :', type(x))          # cupy.ndarray
print('디바이스  :', x.device)         # 어느 GPU에 있는지
print('GPU 결과  :', y[:5])
print('host 전송 :', cp.asnumpy(y)[:5])  # device -> host 복사

타입      : <class 'cupy.ndarray'>
디바이스  : <CUDA Device 0>
GPU 결과  : [2.        2.841471  2.9092975 2.14112   1.2431974]
host 전송 : [2.        2.841471  2.9092975 2.14112   1.2431974]


<a id="6"></a>
## 6. 비동기 타이밍의 함정

GPU 연산은 **비동기**입니다. 파이썬이 `cp.sin(x)`를 호출하면, 실제로는 GPU의 실행 큐(**스트림,
stream**)에 "이 커널을 실행하라"는 명령만 등록하고 곧바로 다음 파이썬 코드로 넘어갑니다. GPU는
자기 큐에 쌓인 작업을 백그라운드에서 순서대로 처리합니다. 그래서 `time.perf_counter()`로
감싸면 **연산이 끝나기 전 시간**이 찍혀 실제보다 빠르게 보입니다(심하면 거의 0에 가깝게 보이기도 합니다).

올바른 측정은 (a) `synchronize()`로 "큐에 쌓인 작업이 모두 끝날 때까지" 기다리거나, (b) `bench`
(CUDA 이벤트를 큐에 함께 넣어 시간을 재는 방식)를 쓰는 것입니다. 스트림이라는 개념 자체는
`06_streams_async`에서 본격적으로 다루며, 이 노트북에서는 "기본 스트림(default stream)"이 있고
그 위에서 명령들이 비동기로 실행된다는 사실만 기억하면 충분합니다.


In [10]:
n = 100_000_000 # n을 변경해보세요.

def work():
    # static 할당
    # a = cp.ones(n, dtype=cp.float32)
    # random number generation
    a = cp.random.random(n, dtype=cp.float32)
    return (a * 1.0001 + 2.0).sum()

# (1) 동기화 없음 — 잘못된 측정 (커널 제출 시간만 잼)
t0 = time.perf_counter(); _ = work(); t1 = time.perf_counter()
print(f'[잘못] sync 없음 : {(t1 - t0) * 1e3:8.3f} ms')

# (2) 직접 동기화 — 올바른 측정
t0 = time.perf_counter(); _ = work(); cp.cuda.Device().synchronize(); t1 = time.perf_counter()
print(f'[정상] sync 포함 : {(t1 - t0) * 1e3:8.3f} ms')

# (3) 권장 — cupyx.profiler.benchmark 래퍼 (워밍업 + 반복 평균)
r = bench(work, n_repeat=20, n_warmup=3)
print(f'[권장] bench GPU : {gpu_ms(r):8.3f} ms (CPU 런치 {cpu_ms(r):.3f} ms)')
print(r)

[잘못] sync 없음 :    0.941 ms
[정상] sync 포함 :    4.319 ms
[권장] bench GPU :    2.635 ms (CPU 런치 0.131 ms)
work                :    CPU:   131.008 us   +/- 29.461 (min:    84.201 / max:   168.582) us     GPU-0:  2635.264 us   +/-  9.899 (min:  2620.416 / max:  2647.040) us


### 6.1 일회성 오버헤드와 워밍업

첫 측정이 유난히 느린 데는 이유가 있습니다(공식 문서가 "One-Time Overheads"로 안내).
- **컨텍스트 초기화**: 프로세스에서 **첫 CUDA 호출** 시 드라이버가 CUDA 컨텍스트를 만드느라 **수 초**가 걸릴 수 있습니다.
- **커널 컴파일(JIT)**: CuPy는 인자의 shape/dtype에 맞춰 커널을 **즉석 컴파일**합니다. 결과는 프로세스 내 캐시 + 디스크(`~/.cupy/kernel_cache`, 환경변수 `CUPY_CACHE_DIR`)에 저장되어 다음 호출부터 빨라집니다.

그래서 측정에는 **워밍업**이 필수입니다(`bench`가 자동 처리). 주피터에서는 전용 매직 `%gpu_timeit`도 편리합니다:

```python
%load_ext cupyx.profiler
%gpu_timeit (cp.random.random(1_000_000, dtype=cp.float32) ** 2).sum()
```

> 실습 팁: `course_utils.bench`의 기본값(`n_warmup=3`)은 이런 일회성 비용을 피하기 위한 최소한의
> 값입니다. 커널 컴파일이 유난히 무거운 연산(예: 처음 보는 shape의 복잡한 `fuse` 함수)이라면
> `n_warmup`을 늘려야 안정적인 측정이 나올 수 있습니다.


<a id="7"></a>
## 7. 전송 비용 (host ↔ device)

`cp.asnumpy()`(device→host)와 `cp.asarray()`(host→device)는 **PCIe 버스를 통한 물리적 복사**라
비용이 큽니다. 일반적인 PCIe Gen4 x16 기준 이론 대역폭은 편도 약 32 GB/s 수준으로, GPU 내부 메모리
대역폭(수백 GB/s~수 TB/s)보다 한 자릿수 이상 느립니다. 게다가 매 전송마다 고정 지연(latency)도 있어,
작은 데이터를 자주 전송하면 대역폭보다 이 고정 지연이 병목이 됩니다(2절의 지연시간 vs 처리량 논의와
같은 맥락입니다).

연산은 GPU에 **머무르게** 하고 전송은 꼭 필요한 순간에만 하세요. 아래는 같은 연산을 '전송 없음 vs
포함'으로 비교합니다. (참고: `cudaHostAlloc`으로 확보하는 **pinned(고정) 메모리**를 쓰면 전송이
더 빨라지는데, 이는 `05_memory_profiling`/`06_streams_async`에서 다룹니다.)


In [11]:
n = 20_000_000  # n을 변경해보세요.
a = cp.random.random(n, dtype=cp.float32)

# 연산만: 결과 스칼라도 GPU에 둠
r_compute = bench(lambda: (a * 2.0 + 1.0).sum(), n_repeat=20)
# 연산 + host 전송: 매번 asnumpy 호출
r_e2e     = bench(lambda: cp.asnumpy((a * 2.0 + 1.0).sum()), n_repeat=20)

print(f'연산만          : {gpu_ms(r_compute):8.3f} ms')
print(f'연산+전송(e2e)  : {gpu_ms(r_e2e):8.3f} ms')
print('=> 반복문 안에서 asnumpy를 호출하면 전송이 병목이 됩니다. (단원 3에서 더 깊이 다룸)')

연산만          :    0.315 ms
연산+전송(e2e)  :    0.337 ms
=> 반복문 안에서 asnumpy를 호출하면 전송이 병목이 됩니다. (단원 3에서 더 깊이 다룸)


<a id="8"></a>
## 8. 연습문제 — NumPy → CuPy 포팅

아래 `feature_cpu`는 입력을 **z-score 표준화**한 뒤 **[0, 1]로 클리핑**합니다.
`feature_gpu`를 CuPy로 완성하세요. 조건:
- CuPy 연산만 사용하고, **마지막까지 GPU에 머무르게** 합니다(중간 `asnumpy` 금지) — 2절 "전송
  최소화" 규칙을 직접 적용해보는 연습입니다.
- 검증 셀의 주석을 해제해 `assert_allclose`가 통과하는지 확인합니다 — 역시 2절 "정확성 검증"
  규칙의 실전 적용입니다.


In [14]:
def feature_cpu(x_np):
    mu = x_np.mean(); sd = x_np.std() + 1e-8
    z = (x_np - mu) / sd
    return np.clip(z, 0.0, 1.0)

def feature_gpu(x_cp):
    # TODO: 위와 동일한 결과를 CuPy로 구현하세요 (cp.* 함수만 사용)
    # raise NotImplementedError
    mu = x_cp.mean(); sd = x_cp.std() + 1e-8
    z = (x_cp - mu) / sd
    return cp.clip(z, 0.0, 1.0)

# 검증 (구현 후 아래 두 줄 주석 해제)
x_np = np.random.randn(1_000_000).astype(np.float32)
ref = feature_cpu(x_np)
out = cp.asnumpy(feature_gpu(cp.asarray(x_np)))
np.testing.assert_allclose(ref, out, rtol=1e-5, atol=1e-5); print('OK')

OK


<details>
<summary>💡 해답 보기</summary>

```python
def feature_gpu(x_cp):
    mu = x_cp.mean(); sd = x_cp.std() + 1e-8
    z = (x_cp - mu) / sd
    return cp.clip(z, 0.0, 1.0)

x_np = np.random.randn(1_000_000).astype(np.float32)
ref = feature_cpu(x_np)
out = cp.asnumpy(feature_gpu(cp.asarray(x_np)))
np.testing.assert_allclose(ref, out, rtol=1e-5, atol=1e-5)
print('OK')
```

포인트: `np.` → `cp.` 치환만으로 동작합니다. `mean/std/clip`이 모두 GPU에서 실행되고,
`asnumpy`는 **검증을 위한 마지막 한 번**만 호출합니다. `rtol=atol=1e-5`처럼 비교적 타이트한
허용오차를 쓸 수 있는 이유는, 이 연산(평균/표준편차/클리핑)이 단순 원소별 연산이라 CPU와 GPU의
연산 순서 차이가 결과에 거의 영향을 주지 않기 때문입니다. 반면 행렬곱·리덕션이 깊게 얽힌 연산은
오차가 더 커질 수 있어 03·04 노트북에서는 더 느슨한 허용오차를 씁니다.
</details>


## 🧪 추가 실험 — 전송 비용과 대역폭

`asnumpy`(device→host) 전송 시간을 크기별로 재고 **유효 대역폭(GB/s)** 을 추정해 보세요.
> 예측 먼저: 크기가 100배 늘면 전송 시간도 100배일까요? 대역폭은 일정할까요?

힌트: 전송 시간은 대략 `시간 ≈ 고정 지연(latency) + 데이터크기 / 대역폭(bandwidth)` 형태의 모델을
따릅니다. 데이터가 작을 때는 고정 지연이 지배적이라 **유효 대역폭이 낮게** 측정되고, 데이터가
충분히 커지면 고정 지연의 비중이 사라지며 **실제 PCIe 대역폭에 근접**합니다. 이 "작을 때 vs 클 때"의
패턴은 이후 커널 성능을 해석할 때도 반복해서 등장하는 사고방식입니다.


In [15]:
for n in [10_000, 100_000, 1_000_000, 10_000_000, 100_000_000]:
    a = cp.random.random(n, dtype=cp.float32)
    r = bench(lambda a=a: cp.asnumpy(a), n_repeat=10, n_warmup=3)
    ms = cpu_ms(r); gbps = (n*4) / (ms/1e3) / 1e9
    print(f'N={n:>12,} | size={n*4/1e6:6.2f} GB | {ms:8.3f} ms | ~{gbps:5.2f} GB/s')
# 관찰: 작은 N은 고정 오버헤드 지배(대역폭 낮게 측정), 큰 N에서 실제 PCIe 대역폭에 근접

N=      10,000 | size=  0.04 GB |    0.032 ms | ~ 1.24 GB/s
N=     100,000 | size=  0.40 GB |    0.160 ms | ~ 2.50 GB/s
N=   1,000,000 | size=  4.00 GB |    1.067 ms | ~ 3.75 GB/s
N=  10,000,000 | size= 40.00 GB |   14.844 ms | ~ 2.69 GB/s
N= 100,000,000 | size=400.00 GB |  144.315 ms | ~ 2.77 GB/s


<a id="9"></a>
## 9. 체크포인트

- [ ] `print_env()`에 GPU 이름과 VRAM이 출력됨
- [ ] CPU(지연시간)와 GPU(처리량)의 차이를 한 문장으로 설명할 수 있음
- [ ] GPU 연산이 **비동기**임을 이해하고, `bench`로 올바르게 측정함
- [ ] `asnumpy`(전송)가 비용이 크다는 것을 수치로 확인함
- [ ] `course_utils`의 `bench`/`gpu_ms`/`cpu_ms`가 각각 무엇을 측정하는지 설명할 수 있음
- [ ] 연습문제 `feature_gpu`가 `assert_allclose`를 통과함

다음 노트북: **`01_benchmark_basics`** — NumPy vs CuPy를 크기별로 벤치마크하고 GPU 가속의 손익분기점을 찾습니다.
